In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

In [178]:
df = pd.read_csv(r"D:\customer-intelligence-engine\data\processed\customer_master.csv")

In [179]:
df.head()

,customer_unique_id,total_orders,total_spend,avg_order_value,first_purchase_date,last_purchase_date,recency_days,avg_review_score,churn
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,5/10/2018 10:56,5/10/2018 10:56,111,5,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,5/7/2018 11:11,5/7/2018 11:11,114,4,0
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,3/10/2017 21:05,3/10/2017 21:05,537,3,1
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,10/12/2017 20:29,10/12/2017 20:29,321,4,0
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,11/14/2017 19:45,11/14/2017 19:45,288,5,0


In [180]:
df.shape

(93357, 9)

In [181]:
df.isnull().sum()[df.isnull().sum()> 0]

Series([], dtype: int64)

In [182]:
# checking data type
df['avg_review_score'].dtype
# its object so we need to convert it into numeric. so,
df['avg_review_score'] =pd.to_numeric( df['avg_review_score'] , errors = 'coerce')
df['first_purchase_date'] = pd.to_datetime(df['first_purchase_date'] )
df['last_purchase_date'] = pd.to_datetime(df['last_purchase_date'] )

In [183]:
# filling null values by mean
df['avg_review_score'] = df['avg_review_score'].fillna(df['avg_review_score']).mean()

In [184]:
# Average Order Value
df['avg_order_value'] = df['total_spend']/ df['total_orders']

In [185]:
df['customer_lifetime_days'] = (df['last_purchase_date'] - df['first_purchase_date'] ).dt.days +1

In [186]:
df.columns

Index(['customer_unique_id', 'total_orders', 'total_spend', 'avg_order_value',
       'first_purchase_date', 'last_purchase_date', 'recency_days',
       'avg_review_score', 'churn', 'customer_lifetime_days'],
      dtype='object')

In [187]:
df['order_frequency'] = df['total_orders'] / df['customer_lifetime_days']

In [188]:
df['engagement_score'] = (df['total_orders'] * 0.4 + df['avg_review_score'] * 0.3 + (1/(df['recency_days'] + 1)) * 0.3)

In [189]:
df['r_score'] = pd.qcut(df["recency_days"].rank(method = 'first'), q = 4, labels = [4,3,2,1])
df['f_score'] = pd.qcut(df["total_orders"].rank(method = 'first'), q = 4, labels = [1,2,3,4])
df['m_score'] = pd.qcut(df["recency_days"].rank(method = 'first'), q = 4, labels = [4,3,2,1])

In [190]:
df['rfm_score'] = df['r_score'].astype('int') + df['f_score'].astype('int') + df['m_score'].astype('int')

In [191]:
df.head()

,customer_unique_id,total_orders,total_spend,avg_order_value,first_purchase_date,last_purchase_date,recency_days,avg_review_score,churn,customer_lifetime_days,order_frequency,engagement_score,r_score,f_score,m_score,rfm_score
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,2018-05-10 10:56:00,2018-05-10 10:56:00,111,4.153703,0,1,1.0,1.648790,4,1,4,9
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,2018-05-07 11:11:00,2018-05-07 11:11:00,114,4.153703,0,1,1.0,1.648720,4,1,4,9
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,2017-03-10 21:05:00,2017-03-10 21:05:00,537,4.153703,1,1,1.0,1.646669,1,1,1,3
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,2017-10-12 20:29:00,2017-10-12 20:29:00,321,4.153703,0,1,1.0,1.647043,2,1,2,5
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,2017-11-14 19:45:00,2017-11-14 19:45:00,288,4.153703,0,1,1.0,1.647149,2,1,2,5


In [192]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93357 entries, 0 to 93356
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   customer_unique_id      93357 non-null  object        
 1   total_orders            93357 non-null  int64         
 2   total_spend             93357 non-null  float64       
 3   avg_order_value         93357 non-null  float64       
 4   first_purchase_date     93357 non-null  datetime64[ns]
 5   last_purchase_date      93357 non-null  datetime64[ns]
 6   recency_days            93357 non-null  int64         
 7   avg_review_score        93357 non-null  float64       
 8   churn                   93357 non-null  int64         
 9   customer_lifetime_days  93357 non-null  int64         
 10  order_frequency         93357 non-null  float64       
 11  engagement_score        93357 non-null  float64       
 12  r_score                 93357 non-null  catego

In [193]:
# dropped non numeric cols 
df = df.drop(columns = ["customer_unique_id" ,"first_purchase_date" , "last_purchase_date", "recency_days"])

In [194]:
# defining x and y
x = df.drop(['churn'], axis =1)
y = df['churn']

In [195]:
# balance of churn data
y.value_counts(normalize = True)

churn
0    0.744647
1    0.255353
Name: proportion, dtype: float64

In [196]:
# df['recency_days'].describe()

In [197]:
x_train, x_test, y_train, y_test = train_test_split(x, y , test_size= 0.2, random_state= 42, stratify= y)

In [198]:
scaler = StandardScaler()
x_train_scaler = scaler.fit_transform(x_train)
x_test_scaler = scaler.transform(x_test)

In [199]:
lr = LogisticRegression(max_iter=1000)
lr.fit(x_train_scaler, y_train)
y_pred = lr.predict(x_test_scaler)
print(classification_report(y_test, y_pred))
print("roc auc score: ",roc_auc_score(y_test, lr.predict_proba(x_test_scaler)[:,1]))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00     13904
           1       1.00      0.98      0.99      4768

    accuracy                           0.99     18672
   macro avg       1.00      0.99      0.99     18672
weighted avg       0.99      0.99      0.99     18672

roc auc score:  0.9969986848939227


In [202]:
y.value_counts()

churn
0    69518
1    23839
Name: count, dtype: int64